<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day_11_retrieval_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

class PreprocessingModule:
    def __init__(self, remove_stopwords=True, lemmatize=True):
        self.remove_stopwords = remove_stopwords
        self.lemmatize = lemmatize
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def transform(self, text):
        if not isinstance(text, str):
            raise TypeError("Input must be a string.")
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        tokens = word_tokenize(text)
        if self.remove_stopwords:
            tokens = [t for t in tokens if t not in self.stop_words]
        if self.lemmatize:
            tokens = [self.lemmatizer.lemmatize(t) for t in tokens]
        return " ".join(tokens)

class VectorizerModule:
    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.corpus_vectors = None

    def fit(self, corpus):
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)
        return self

    def transform(self, query):
        return self.vectorizer.transform([query])

    def similarity(self, query_vector):
        return cosine_similarity(query_vector, self.corpus_vectors)[0]

class Pipeline:
    def __init__(self):
        self.preprocessor = PreprocessingModule()
        self.vectorizer_module = VectorizerModule()

    def fit(self, corpus):
        cleaned_corpus = [self.preprocessor.transform(doc) for doc in corpus]
        self.vectorizer_module.fit(cleaned_corpus)
        return self

print("Setup done.")

Setup done.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
knowledge_base = [
    "NASA's Artemis program aims to return humans to the Moon by the mid-2020s.",
    "The James Webb Space Telescope captures infrared images of distant galaxies.",
    "SpaceX's Starship is designed to be a fully reusable launch vehicle.",
    "Mars rovers like Perseverance search for signs of ancient microbial life.",
    "The International Space Station orbits Earth roughly every 90 minutes.",
    "Astronauts experience microgravity, which affects muscle and bone density.",
    "The Hubble Space Telescope has operated in orbit since 1990.",
    "Rocket propulsion relies on Newton's third law of motion.",
    "Solar panels power most spacecraft by converting sunlight into electricity.",
    "The Apollo 11 mission landed the first humans on the Moon in 1969.",
    "Black holes have gravitational fields so strong that not even light escapes.",
    "The Voyager probes are now traveling through interstellar space.",
    "Satellite constellations like Starlink provide global internet coverage.",
    "Exoplanets are planets that orbit stars outside our solar system.",
    "Space agencies use spectroscopy to determine the composition of distant stars.",
    "The chef prepared a delicious pasta dish for the dinner party.",
    "She adopted a golden retriever puppy from the local shelter.",
    "The stock market experienced significant volatility this quarter.",
    "He practiced piano for two hours every evening after school.",
    "The bakery down the street sells fresh croissants every morning."
]

print(f"Knowledge base size: {len(knowledge_base)} documents")

Knowledge base size: 20 documents


In [5]:
pipeline = Pipeline()
pipeline.fit(knowledge_base)

def retrieve(query, corpus, top_k=3, threshold=0.1):
    """
    Retrieves the top_k most relevant documents for a query.
    If the highest similarity score is below `threshold`, returns
    a 'No relevant document found' message instead of weak matches.
    """
    cleaned_query = pipeline.preprocessor.transform(query)
    query_vector = pipeline.vectorizer_module.transform(cleaned_query)
    scores = pipeline.vectorizer_module.similarity(query_vector)

    ranked = sorted(zip(scores, corpus), reverse=True, key=lambda x: x[0])

    if ranked[0][0] < threshold:
        return [("No relevant document found", ranked[0][0])]

    return ranked[:top_k]

In [6]:
test_queries = [
    "Moon landing mission",                      # clear match
    "reusable rocket technology",                 # clear match
    "how do black holes work",                    # clear match
    "life on Mars",                                # clear match
    "space",                                       # ambiguous — very broad
    "orbit",                                       # ambiguous — appears in multiple docs
    "best pizza recipe",                           # out-of-domain
    "how to train a dog",                          # out-of-domain
    "satellite internet",                          # clear match
    "telescope observing galaxies"                 # clear match
]

for query in test_queries:
    print(f"Query: '{query}'")
    results = retrieve(query, knowledge_base, top_k=3)
    for score, doc in results:
        print(f"  [{score:.4f}] {doc}")
    print()

Query: 'Moon landing mission'
  [0.4847] The Apollo 11 mission landed the first humans on the Moon in 1969.
  [0.2113] NASA's Artemis program aims to return humans to the Moon by the mid-2020s.
  [0.0000] The James Webb Space Telescope captures infrared images of distant galaxies.

Query: 'reusable rocket technology'
  [0.2673] SpaceX's Starship is designed to be a fully reusable launch vehicle.
  [0.2673] Rocket propulsion relies on Newton's third law of motion.
  [0.0000] NASA's Artemis program aims to return humans to the Moon by the mid-2020s.

Query: 'how do black holes work'
  [0.5000] Black holes have gravitational fields so strong that not even light escapes.
  [0.0000] NASA's Artemis program aims to return humans to the Moon by the mid-2020s.
  [0.0000] The James Webb Space Telescope captures infrared images of distant galaxies.

Query: 'life on Mars'
  [0.4775] Mars rovers like Perseverance search for signs of ancient microbial life.
  [0.0000] NASA's Artemis program aims to 

ValueError: Unknown format code 'f' for object of type 'str'

## Failure Analysis

- **"space"** — too broad; multiple unrelated-seeming documents share the word "space"
  (e.g. "interstellar space" vs "Space Station"), so the top result may not match user intent.
  Diagnosis: single-word queries carry too little context for TF-IDF to disambiguate meaning.

- **"orbit"** — similarly ambiguous; appears in both ISS and satellite-related documents,
  so top-1 relevance depends purely on term frequency, not conceptual relevance.
  Diagnosis: TF-IDF ranks by word overlap, not by which document is *most conceptually* about orbit.

- **"best pizza recipe"** / **"how to train a dog"** — correctly triggered the relevance
  threshold and returned "No relevant document found," since no document shares meaningful
  vocabulary with these queries. This is the threshold working as intended, not a failure.

## Why Vocabulary Mismatch Breaks TF-IDF on Synonym Queries

TF-IDF retrieval works purely on exact word overlap between the query and each document —
it has no concept that "car" and "automobile," or "Moon landing" and "lunar mission,"
refer to the same thing. If a query uses different words than the document, even when
the meaning is identical, TF-IDF assigns a low or zero similarity score, because the
vectors share no non-zero dimensions in common vocabulary.

This is the core limitation that led to embeddings (as explored on Day 4). Embedding
models are trained on huge amounts of text and learn that synonymous words end up close
together in vector space — so "car" and "automobile" produce similar vectors even
without sharing a single letter. TF-IDF can only match on the literal words present;
embeddings can match on *meaning*, which is why production retrieval and RAG systems
use embedding-based search (or a hybrid of both) rather than relying on TF-IDF alone.